In [1]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters.character import RecursiveCharacterTextSplitter
from sklearn.metrics.pairwise import cosine_similarity
import warnings
import re
import ollama

/tmp/ipykernel_37655/748431923.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, TextLoader


In [2]:
loader = PyPDFLoader("data/10201979.pdf")
loader = PyPDFLoader("data/Mr. Himanshu Barhaiya - Advances in Retrieval-Augmented Generation for Scientific and Technical QA A Review [2026].pdf")
pages = loader.load()

#### text Cleaning

In [3]:
def cleanText(text): 
    cleaned_text = re.sub(r'\s+', ' ', text) # clears the spacess
    cleaned_text = cleaned_text.replace('-', " ")
    cleaned_text = cleaned_text.replace(' -', " ")
    cleaned_text = cleaned_text.replace('•', ";")
    cleaned_text = cleaned_text.replace("   ", " ")
    return cleaned_text

In [4]:
for page in pages: 
    page.page_content = cleanText(page.page_content) 

In [17]:
pages[: 10]

[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-05-05T22:07:15+05:30', 'author': 'TechnoScience Academy;http://technoscienceacademy.com', 'keywords': 'IJSRCSEIT; http:/ijsrcseit.com', 'moddate': '2026-05-05T22:07:15+05:30', 'subject': 'Research Paper', 'title': 'International Journal of Scientific Research in Computer Science, Engineering and Information Technology', 'source': 'data/Mr. Himanshu Barhaiya - Advances in Retrieval-Augmented Generation for Scientific and Technical QA A Review [2026].pdf', 'total_pages': 12, 'page': 0, 'page_label': '1'}, page_content='Copyright © 202 6 The Author(s) : This is an open access article under the CC BY license (http://creativecommons.org/licenses/by/4.0/) International Journal of Scientific Research in Computer Science, Engineering and Information Technology Peer Reviewed and Refereed International Scientific Research Journal ISSN : 2456 3307 Available Online at : ijsrcseit.com do

#### Chunking
- splitting your text information into smaller sizes

In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 200, chunk_overlap= 100) # created an instance of the text splitter

In [6]:
chunks = splitter.split_documents(pages)

In [18]:
chunks[0:10]

[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-05-05T22:07:15+05:30', 'author': 'TechnoScience Academy;http://technoscienceacademy.com', 'keywords': 'IJSRCSEIT; http:/ijsrcseit.com', 'moddate': '2026-05-05T22:07:15+05:30', 'subject': 'Research Paper', 'title': 'International Journal of Scientific Research in Computer Science, Engineering and Information Technology', 'source': 'data/Mr. Himanshu Barhaiya - Advances in Retrieval-Augmented Generation for Scientific and Technical QA A Review [2026].pdf', 'total_pages': 12, 'page': 0, 'page_label': '1'}, page_content='Copyright © 202 6 The Author(s) : This is an open access article under the CC BY license (http://creativecommons.org/licenses/by/4.0/) International Journal of Scientific Research in Computer Science,'),
 Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-05-05T22:07:15+05:30', 'author': 'TechnoScience

In [19]:
len(chunks)

405

### Create Embeddings:
- Create embeddings from our chunks
- perform a search:
    - get user input
    - calculate cosine similarity (check for similarity in text/ information)

In [20]:
# functions & global variables 

def createEmbedding(text):
    response = ollama.embed(
    model='nomic-embed-text-v2-moe',
    input=text,
    )
    return response.embeddings

In [23]:
sample_text = "this is a blue car!"
print(createEmbedding(sample_text))

[[-0.010901414, -0.009878994, -0.10206994, 0.025146862, -0.029017122, -0.018902695, -0.026545098, -0.03951932, -0.021458616, 0.02708927, 0.021827163, -0.08983775, 0.037547894, 0.03862853, 0.02944821, -0.016738234, 0.022206597, -0.014421699, -0.013778293, 0.007764854, 0.065272786, -0.0096959155, 0.017373862, -0.06427167, 0.056987867, -0.0032896744, 0.031153461, 0.005081759, 0.042577192, -0.04700088, 0.015497484, -0.03952028, -0.057451714, 0.06289506, -0.07147578, 0.05663965, -0.021548716, 0.010057831, 0.004936667, 0.028589593, 0.006044981, 0.0025699749, 0.046583567, 0.058653835, 0.020789199, -0.01923741, 0.009776124, -0.011004501, -0.015905574, -0.015037399, 0.052893896, -0.078270756, 0.022161713, -0.08693023, 0.041561525, -0.017274125, 0.026195088, -0.043717206, 0.02374817, -0.037567746, 0.012711958, -0.00049154134, -0.000664973, -0.037378713, -0.04365364, -0.04755528, 0.03867042, -0.012658174, 0.029582467, 0.009293176, -0.039772976, 0.035374735, -0.024829699, 0.04648463, -0.018639453,

In [ ]:
doc_embeddings = [] # this is our list of embeddings from out document

for chunk in chunks:
    chunk_embedding = createEmbedding(chunk.page_content)
    doc_embeddings.append(chunk_embedding) 

In [24]:
len(doc_embeddings)

405

- user asks a question
- embedd the question 
- check for similarity in the document  

In [ ]:
user_query = input("ask me anything about this doc...")

In [28]:
print(user_query)

what is the RAG  model?


In [27]:
user_query_embeddding = createEmbedding(user_query)

In [29]:
print(user_query_embeddding)

[[0.03596005, -0.07062215, -0.037696034, -0.0055717826, 0.033673048, -0.04018488, 0.0018334462, -0.017024627, -0.050206553, 0.030212427, -0.028990619, -0.045147225, -0.023806611, 0.014902039, -0.006569416, -0.04558168, 0.015312845, 0.010345008, 0.0135051925, 0.02333032, -0.03464431, 0.026333882, 0.0056496914, -0.028721606, 0.033138942, 0.002761118, -0.05661325, -0.005022051, 0.007374197, 0.04581732, -0.019784665, -0.014430813, -0.0074599893, 0.053319342, -0.00862045, 0.029752456, -0.0067633176, -0.021389246, 0.008265082, 0.00010303403, 0.01884163, 0.037268933, 0.04446289, -0.0009950832, 0.050941348, -0.05452661, -0.07831083, -0.013394604, -0.00683497, -0.039807748, 0.025613267, -0.029788023, -0.011113522, -0.038029127, -0.06324102, -0.06799142, -0.047971617, -0.04799917, 0.027156712, -0.026289873, -0.032669287, -0.04438442, -0.041372564, 0.063703924, -0.042168435, 0.07200018, 0.053574298, 0.04863703, 0.015571414, 0.043841433, -0.06021627, -0.008917362, 0.007310821, -0.0037752227, 0.012

In [34]:
# - loop through the doc_embeedding 
# - calculate the cosine similarity score 
# - return the items with highest score 

above_zero_point_six = []

for index, doc_embedding in enumerate(doc_embeddings):
    score = cosine_similarity(doc_embedding, user_query_embeddding)
    if score >= 0.6:
        result = (index, score)
        above_zero_point_six.append(result)
        # print(score, index)
    

In [45]:
above_zero_point_six

[(27, array([[0.60503961]])),
 (54, array([[0.66094132]])),
 (63, array([[0.63030478]])),
 (66, array([[0.60492278]])),
 (67, array([[0.62228588]])),
 (70, array([[0.61341911]])),
 (117, array([[0.62823035]])),
 (118, array([[0.61580554]])),
 (175, array([[0.61581443]])),
 (246, array([[0.61094932]])),
 (279, array([[0.61763213]])),
 (283, array([[0.60688908]]))]

In [47]:
largest =  max(above_zero_point_six, key=lambda x: x[1])
largest

(54, array([[0.66094132]]))

In [50]:
response_to_client = chunks[largest[0]]
print(response_to_client.page_content)

(RAG) RAG is a brand  new mixed system that claims to be able to fix the issues with pure generative models. There is a system that gets relevant papers or information from an outside source of


In [ ]:
(RAG) RAG is a brand  new mixed system that claims to be able to fix the issues with pure generative models. 
There is a system that gets relevant papers or information from an outside source of